# Multi agente simples

Neste exemplo, vamos criar um multi agente com um escritor e um revisor. O escritor irá gerar um texto com base em um prompt, e o revisor irá revisar o texto gerado pelo escritor, sugerindo melhorias.

## Carregando as dependências

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from random import randint
from typing import Annotated
from typing import cast

from agent_framework import tool
from pydantic import Field

## Carregando variáveis de ambiente

In [ ]:
from dotenv import load_dotenv
load_dotenv(override=True)

In [ ]:
# print(f"AZURE_OPENAI_ENDPOINT: {os.getenv('AZURE_OPENAI_ENDPOINT')}")
# print(f"AZURE_OPENAI_CHAT_DEPLOYMENT_NAME: {os.getenv('AZURE_OPENAI_CHAT_DEPLOYMENT_NAME')}")
# print(f"AZURE_OPENAI_API_VERSION: {os.getenv('AZURE_OPENAI_API_VERSION')}")

## Criando o cliente

Aqui mostramos algumas maneiras de inicializar o cliente do agente, usando diferentes tipos de chaves de API. O cliente é necessário para que o agente possa acessar as ferramentas e realizar suas tarefas.

In [ ]:
from agent_framework.azure import AzureOpenAIChatClient

# Usando variáveis de ambiente
# Set AZURE_OPENAI_ENDPOINT=""
# Set AZURE_OPENAI_CHAT_DEPLOYMENT_NAME=""
# Set AZURE_OPENAI_API_VERSION=""
# Set AZURE_OPENAI_API_KEY=""
client = AzureOpenAIChatClient()

# # Ou passando os parâmetros diretamente
# client = AzureOpenAIChatClient(
#     endpoint="",
#     deployment_name="",
#     api_key=""
# )

# # Ou carregando a partir de um arquivo .env
# client = AzureOpenAIChatClient(
#     env_file_path="path/to/.env"
# )

## Criando os agentes

In [ ]:
writer_agent_1 = client.as_agent(
    instructions=(
        "Você é um excelente escritor de conteúdo. Você cria novos conteúdos e edita conteúdos com base no feedback recebido."
    ),
    name="writer_1",
)

reviewer_agent = client.as_agent(
    instructions=(
        "Você é um excelente revisor de conteúdo. "
        "Forneça feedback acionável para o escritor sobre o conteúdo fornecido. "
        "Forneça o feedback da maneira mais concisa possível."
    ),
    name="reviewer",
)

writer_agent_2 = client.as_agent(
    instructions=(
        "Você é um excelente escritor de conteúdo. Você cria novos conteúdos e edita conteúdos com base no feedback recebido."
    ),
    name="writer_2",
)

## Criando o workflow

In [ ]:
from agent_framework import AgentResponse, WorkflowBuilder
from agent_framework.orchestrations import SequentialBuilder

# ============================================================
# OPÇÃO 1: SequentialBuilder - Fluxo Linear Simples
# ============================================================
# Fluxo: writer → reviewer → FIM
# Cada agente executa apenas UMA vez (sem iterações)
# Vantagem: Simples e direto
# Desvantagem: Sem ciclo de refinamento

# workflow = SequentialBuilder(participants=[writer_agent, reviewer_agent]).build()


# ============================================================
# OPÇÃO 2: WorkflowBuilder com Ciclo Limitado
# ============================================================
# Fluxo: writer → reviewer → writer → FIM
# Permite UMA rodada de feedback e refinamento
# ATENÇÃO: Sem add_edge final, cria loop infinito!

workflow = (
    WorkflowBuilder(start_executor=writer_agent_1, max_iterations=3)
    .add_edge(writer_agent_1, reviewer_agent)  # 1ª execução: Escritor → Revisor
    .add_edge(reviewer_agent, writer_agent_2)  # 2ª execução: Revisor → Escritor (com feedback)
    .build()
)


# ============================================================
# OPÇÃO 3: Reflection Pattern - Ciclo até Aprovação
# ============================================================
# Fluxo: worker ↔ reviewer (ciclo automático até aprovação)
# Worker gera → Reviewer avalia → Se não aprovado: Worker refina
# Requer executores customizados (ver: _other/agents/workflow_as_agent_reflection_pattern.py)
# Vantagem: Refinamento automático com controle de qualidade
# Complexidade: Maior (requer Pydantic e handlers customizados)

## Executando o workflow

In [ ]:
# Executa o workflow com a mensagem inicial do usuário
events = await workflow.run("Crie um slogan para um novo SUV elétrico que seja acessível e divertido de dirigir.")

In [ ]:
outputs = events.get_outputs()

# As saídas são AgentResponse de cada agente no workflow
outputs = cast(list[AgentResponse], outputs)
for output in outputs:
    print(f"{output.messages[0].author_name}: {output.text}\n")

In [ ]:
# Sumarize o estado final (e.g., COMPLETED)
print("Final state:", events.get_final_state())

## Usando o workflow como um agente reutilizável (Opcional)

In [ ]:
# Você pode transformar o workflow em um agente reutilizável
# Isso permite usar o workflow como um "super agente" em outros contextos

# agent = workflow.as_agent(name="WriterReviewerAgent")
# response = await agent.run("Crie um slogan para um notebook gamer de alto desempenho.")

# # Exibe a conversação completa
# if response.messages:
#     print("\n===== Conversação =====")
#     for i, msg in enumerate(response.messages, start=1):
#         name = msg.author_name or msg.role
#         print(f"{'-' * 60}\n{i:02d} [{name}]\n{msg.text}")